<a href="https://colab.research.google.com/github/1816x/Algoritmos-Aprendizaje-Automatico/blob/main/Practica_Tema_13_pt2_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practica Tema 13 Parte 2

**Clase:** Fundamentos Algoritmos de Aprendizaje Automatico

**Tema:** MLOps, Monitoreo de Drift y Caso Churn

## reto 1. entrenamiento con validacion cruzada para churn

entrenamos un modelo de churn y usamos validación cruzada con 5 particiones. Esto sirve para comprobar que el modelo no funciona bien solo en una división específica de los datos, sino que mantiene un rendimiento relativamente estable.

In [7]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline as pipeline
from sklearn.preprocessing import StandardScaler as standard_scaler
from sklearn.linear_model import LogisticRegression as logistic_regression
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

# crear un conjunto de datos de clasificacion binaria simulado
x, y = make_classification(
    n_samples=1000,
    n_features=5,
    random_state=42
)

# dividir los datos en conjuntos de entrenamiento y prueba independientes
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

# crear una pipeline que escala los datos y entrena regresion logistica
pipe_churn = pipeline([
    ("scaler", standard_scaler()),
    ("clf", logistic_regression(max_iter=1000))
])

# entrenar el modelo con los datos de entrenamiento
pipe_churn.fit(x_train, y_train)

# evaluar la estabilidad con validacion cruzada de 5 folds usando roc-auc
cv_auc = cross_val_score(
    pipe_churn,
    x_train,
    y_train,
    cv=5,
    scoring="roc_auc"
)

# calcular el rendimiento promedio y la variabilidad entre los folds
cv_auc_mean = cv_auc.mean()
cv_auc_std = cv_auc.std(ddof=1)

print(f"roc-auc cv mean: {cv_auc_mean:.4f}")
print(f"roc-auc cv std: {cv_auc_std:.4f}")

roc-auc cv mean: 0.9233
roc-auc cv std: 0.0397


La media muestra el rendimiento promedio del modelo en las cinco particiones. La desviacion estandar indica que tanto cambia el resultado entre cada fold; mientras mas pequena sea, mas estable es el modelo.

## reto 2. auditoria de estabilidad fuera de muestra

después probamos el modelo con x_test, que son datos que no usó para entrenar. Calculamos accuracy y ROC-AUC. La idea es verificar si el buen resultado del entrenamiento también se mantiene con datos nuevos.

In [8]:
# obtener la probabilidad predicha de la clase positiva
test_probabilities = pipe_churn.predict_proba(x_test)[:, 1]

# convertir probabilidades en predicciones binarias usando un umbral de 0.50
test_predictions = (test_probabilities >= 0.50).astype(int)

# calcular la precision en datos no vistos
test_accuracy = accuracy_score(y_test, test_predictions)

# calcular roc-auc como una metrica de rendimiento adicional
test_auc = roc_auc_score(y_test, test_probabilities)

print(f"test accuracy: {test_accuracy:.4f}")
print(f"test roc-auc: {test_auc:.4f}")

test accuracy: 0.8850
test roc-auc: 0.9503


Accuracy indica el porcentaje total de predicciones correctas. Sin embargo, en datos desbalanceados no siempre es suficiente porque un modelo puede acertar mucho en la clase mayoritaria y aun asi tener mal desempeno en la clase minoritaria.

## reto 3. deteccion de data/concept drift por comparacion de ventanas

simulamos que el modelo ya está en producción. d
dividimos las predicciones en dos ventanas: una base y una reciente.
después comparamos cuántas predicciones positivas genera en cada una.
si cambia demasiado, puede significar que los datos o el comportamiento real están cambiando; eso es lo que buscamos detectar como drift.

In [9]:
# usar las primeras 100 probabilidades como ventana base
base_window = test_probabilities[:100]

# usar las siguientes 100 probabilidades como ventana reciente
recent_window = test_probabilities[100:200]

# calcular la tasa de prediccion positiva en la ventana base
base_positive_rate = np.mean(base_window >= 0.50)

# calcular la tasa de prediccion positiva en la ventana reciente
recent_positive_rate = np.mean(recent_window >= 0.50)

# calcular la diferencia absoluta entre ambas ventanas
drift_gap = abs(recent_positive_rate - base_positive_rate)

print(f"base positive rate: {base_positive_rate:.4f}")
print(f"recent positive rate: {recent_positive_rate:.4f}")
print(f"absolute drift gap: {drift_gap:.4f}")

base positive rate: 0.4600
recent positive rate: 0.4800
absolute drift gap: 0.0200


La brecha compara cuanto cambio la proporcion de predicciones positivas entre ambas ventanas. Si el cambio es grande, puede ser una senal de drift en los datos o en el comportamiento del modelo.

## reto 4. semaforo operativo para monitoreo

convertimos ese cambio en una regla práctica. Si la diferencia es pequeña, solo monitoreamos. Si aumenta, generamos una alerta. Si es muy grande, lo tratamos como incidente y consideramos revisar, reentrenar o incluso regresar a una versión anterior del modelo.

In [10]:
# asignar un estado segun la brecha de drift
if drift_gap < 0.05:
    status = "observation"
    action = "continue monitoring the model without intervention"
elif drift_gap < 0.15:
    status = "alert"
    action = "review recent data, metrics, and possible distribution changes"
else:
    status = "incident"
    action = "escalate the issue and consider retraining or rollback"

print("status:", status)
print("recommended action:", action)

status: observation
recommended action: continue monitoring the model without intervention


**Interpretacion del semaforo:**

- **Observacion:** el cambio es pequeno, por lo que solo se sigue monitoreando.
- **Alerta:** el cambio ya es relevante y se deben revisar datos, metricas y posibles cambios de distribucion.
- **Incidente:** el cambio es alto y se debe investigar la causa, escalar el problema y evaluar reentrenamiento o rollback.

## conclusion

la actividad enseña que en MLOps no basta con entrenar un modelo y dejarlo corriendo. Hay que comprobar que sea estable, medir cómo se comporta con datos nuevos y detectar cuándo empieza a cambiar su comportamiento.